# 完整示例：从配置到运行

本教程展示如何使用 ADLab 的完整流程。

## 目录
1. [创建模板程序](#1-创建模板程序)
2. [创建任务描述](#2-创建任务描述)
3. [创建Evaluator](#3-创建Evaluator)
4. [创建YAML配置](#4-创建YAML配置)
5. [运行搜索](#5-运行搜索)

## 1. 创建模板程序

In [ ]:
# 模板程序是算法的骨架，包含函数签名和部分实现

template_code = '''
from typing import List

def sort_list(lst: List[int]) -> List[int]:
    """
    Sort a list of integers in ascending order.

    Args:
        lst: A list of integers to sort

    Returns:
        A new list with integers sorted in ascending order
    """
    # TODO: Implement the sorting algorithm
    # Your code here
    pass
'''

# 保存模板程序
with open('template_sorting.py', 'w') as f:
    f.write(template_code)

print("模板程序已保存到 template_sorting.py")

## 2. 创建任务描述

In [ ]:
# 任务描述告诉 LLM 要解决什么问题

task_description = '''
You are an expert programmer. Your task is to implement a sorting algorithm.

Requirements:
1. The function should take a list of integers
2. Return a new list sorted in ascending order
3. Do not modify the original list
4. Handle edge cases: empty list, single element

Example:
Input: [3, 1, 4, 1, 5]
Output: [1, 1, 3, 4, 5]

Important:
- Do not use built-in sorted() or list.sort()
- Implement your own sorting algorithm (e.g., bubble sort, quick sort)
'''

# 保存任务描述
with open('task_description.txt', 'w') as f:
    f.write(task_description)

print("任务描述已保存到 task_description.txt")

## 3. 创建Evaluator

In [ ]:
# 创建自定义 Evaluator

evaluator_code = '''
import random
import subprocess
import tempfile
from algodisco.base.evaluator import Evaluator, EvalResult

class SortingEvaluator(Evaluator):
    """评估排序算法的正确性"""
    
    def __init__(self, num_tests=100):
        self.num_tests = num_tests
        self.test_cases = self._generate_test_cases()
    
    def _generate_test_cases(self):
        """生成随机测试用例"""
        cases = []
        for _ in range(self.num_tests):
            n = random.randint(0, 20)
            case = random.sample(range(-100, 100), n)
            expected = sorted(case)
            cases.append((case, expected))
        return cases
    
    def evaluate_program(self, program_str: str) -> EvalResult:
        """评估程序"""
        try:
            # 写入临时文件
            with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
                f.write(program_str)
                f.flush()
                temp_path = f.name
            
            # 测试
            correct = 0
            for inputs, expected in self.test_cases[:20]:
                test_code = f"""
import sys
sys.path.insert(0, '.')
exec(open("{temp_path}").read())
result = sort_list({inputs})
print(result == {expected})
"""
                exec_result = subprocess.run(
                    ['python', '-c', test_code],
                    capture_output=True,
                    timeout=5
                )
                if exec_result.returncode == 0 and 'True' in exec_result.stdout.decode():
                    correct += 1
            
            score = correct / min(20, len(self.test_cases))
            
            return {
                "score": score,
                "execution_time": 0.0,
                "error_msg": None
            }
            
        except subprocess.TimeoutExpired:
            return {
                "score": 0.0,
                "execution_time": 5.0,
                "error_msg": "Timeout"
            }
        except Exception as e:
            return {
                "score": 0.0,
                "execution_time": 0.0,
                "error_msg": str(e)[:200]
            }
'''

with open('sorting_evaluator.py', 'w') as f:
    f.write(evaluator_code)

print("Evaluator 已保存到 sorting_evaluator.py")

## 4. 创建YAML配置

In [ ]:
# YAML 配置文件

yaml_config = '''
# 方法配置
method:
  template_program_path: "template_sorting.py"
  task_description_path: "task_description.txt"
  language: "python"
  num_samplers: 2
  num_evaluators: 2
  examples_per_prompt: 2
  samples_per_prompt: 2
  max_samples: 100
  llm_max_tokens: 1024
  llm_timeout_seconds: 60
  db_num_islands: 5
  db_reset_period: 7200
  keep_metadata_keys:
    - sample_time
    - eval_time
    - execution_time
    - error_msg

# LLM 配置
llm:
  class_path: "algodisco.providers.llm.openai_api.OpenAIAPI"
  kwargs:
    model: "gpt-3.5-turbo"
    api_key: "${OPENAI_API_KEY}"  # 使用环境变量
    base_url: "https://api.openai.com/v1"

# Evaluator 配置
evaluator:
  class_path: "sorting_evaluator.SortingEvaluator"
  kwargs:
    num_tests: 100

# 日志配置
logger:
  class_path: "algodisco.providers.logger.pickle_logger.BasePickleLogger"
  kwargs:
    logdir: "logs/sorting_search"
'''

with open('config.yaml', 'w') as f:
    f.write(yaml_config)

print("YAML 配置已保存到 config.yaml")

## 5. 运行搜索

In [ ]:
# 运行搜索

# 方式1: 使用命令行
# python algodisco/methods/funsearch/main_funsearch.py --config config.yaml

# 方式2: 使用 Python 代码

import os
os.environ["OPENAI_API_KEY"] = "your-api-key"

# 导入需要的模块
from algodisco.methods.funsearch.search import FunSearch

# 注意: 实际运行需要确保:
# 1. 模板文件存在
# 2. 任务描述文件存在
# 3. Evaluator 可以被导入
# 4. LLM API key 正确设置

# 示例运行（伪代码）:
# search = FunSearch(
#     template_program_path="template_sorting.py",
#     task_description_path="task_description.txt",
#     llm=OpenAIAPI(model="gpt-3.5-turbo"),
#     evaluator=SortingEvaluator(num_tests=100),
#     logdir="logs/sorting_search"
# )
# search.run(max_samples=100)

print("运行命令:")
print("python algodisco/methods/funsearch/main_funsearch.py --config config.yaml")

## 总结

运行 ADLab 搜索的完整流程:

1. **准备模板程序** - 定义函数骨架
2. **准备任务描述** - 告诉 LLM 要做什么
3. **创建 Evaluator** - 评估生成的代码
4. **编写 YAML 配置** - 配置所有组件
5. **运行搜索** - 执行搜索并获取结果
